# Extração de características de textura — TP1 E2

Esta rotina obtém a base RSNA, utiliza os IDs congelados de treino/validação/teste e gera matrizes de textura com 20 características (LBP e GLCM) na ordem de cada partição. O pré-processamento comum converte para cinza, redimensiona para 224×224 com Lanczos e normaliza por 255.

A execução requer uma sessão do Google Colab conectada. Ao concluir, salvar `matrizes_textura_RSNA_4000.zip`, que contém matrizes, metadados alinhados, IDs e documentação da execução. O compartilhamento do pacote é uma etapa externa ao processamento.

A separação dos IDs é por imagem; a base utilizada não oferece chave verificável de paciente. Os dados brutos permanecem fora do Git. As saídas registradas correspondem à execução anterior; esta revisão altera somente a documentação e mensagens de orientação.


In [1]:
"""Execute esta célula inteira no Google Colab."""
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
import json
import platform
import runpy
import shutil
import subprocess
import sys
import tempfile

REPO_URL = "https://github.com/RaulFH0/tp1-e2-bone-age.git"
COMMIT = "888551893ef0667ea9d55835d5f0c2c47d6568bf"
DATASET = "kmader/rsna-bone-age/versions/2"
TAMANHOS = {"train": 2800, "val": 600, "test": 600}

print("0% — Preparando uma execução independente das células antigas.", flush=True)
trabalho = Path(tempfile.mkdtemp(prefix="entrega_textura_", dir="/content"))
repo_textura = trabalho / "repo"
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(repo_textura)], check=True)
subprocess.run(
    ["git", "-C", str(repo_textura), "checkout", "--quiet", "--detach", COMMIT],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(repo_textura / "requirements.txt"), "kagglehub==1.0.2", "tqdm"],
    check=True,
)

import numpy as np
import pandas as pd
import kagglehub
from tqdm.auto import tqdm


def sha256(caminho):
    return hashlib.sha256(Path(caminho).read_bytes()).hexdigest()


def ler_ids(caminho):
    tabela = pd.read_csv(caminho, dtype={"id": str}, keep_default_na=False)
    if tabela.columns.tolist() != ["id"]:
        raise ValueError(f"Colunas inesperadas em {caminho}.")
    if not tabela["id"].str.fullmatch(r"[0-9]+").all():
        raise ValueError(f"ID vazio ou inválido em {caminho}.")
    ids = tabela["id"].tolist()
    if len(ids) != len(set(ids)):
        raise ValueError(f"ID repetido em {caminho}.")
    return ids


# Lê a partição congelada. Não recria amostra nem divisões.
splits = {
    nome: ler_ids(repo_textura / "data/splits" / f"{nome}_ids.csv")
    for nome in TAMANHOS
}
for nome, ids in splits.items():
    if len(ids) != TAMANHOS[nome]:
        raise ValueError(f"Tamanho incorreto no conjunto {nome}.")
amostra = ler_ids(repo_textura / "data/splits/sample_ids.csv")
todos_ids = [id_ for ids in splits.values() for id_ in ids]
if (len(amostra) != 4000 or len(todos_ids) != 4000
        or len(set(todos_ids)) != 4000 or set(todos_ids) != set(amostra)):
    raise ValueError("Os splits precisam cobrir exatamente os 4.000 IDs da amostra.")
print("25% — Código e IDs congelados preparados.", flush=True)

# Sem path de pasta: funciona tanto no cache quanto no download normal.
pasta_dataset = Path(kagglehub.dataset_download(DATASET))
candidatos = list(pasta_dataset.rglob("1377.png"))
if len(candidatos) != 1:
    raise RuntimeError(f"Pasta de treino não identificada: {candidatos}")
pasta_imagens = candidatos[0].parent
quantidade = sum(1 for _ in pasta_imagens.glob("*.png"))
if quantidade != 12611:
    raise ValueError(f"Esperávamos 12.611 PNG de treino; encontramos {quantidade}.")
ausentes = [id_ for id_ in todos_ids if not (pasta_imagens / f"{id_}.png").is_file()]
if ausentes:
    raise FileNotFoundError(f"Imagens ausentes; exemplos: {ausentes[:10]}")

# Obtém rótulos do mesmo dataset, para entregar y e sexo na mesma ordem.
candidatos_csv = list(pasta_dataset.rglob("boneage-training-dataset.csv"))
if len(candidatos_csv) == 1:
    csv_rotulos = candidatos_csv[0]
else:
    csv_rotulos = Path(kagglehub.dataset_download(DATASET, path="boneage-training-dataset.csv"))
rotulos = pd.read_csv(csv_rotulos, dtype={"id": str}, keep_default_na=False)
if len(rotulos) != 12611 or not {"id", "boneage", "male"}.issubset(rotulos.columns):
    raise ValueError("Tabela de rótulos com tamanho ou colunas inesperados.")
if rotulos["id"].duplicated().any():
    raise ValueError("Há IDs duplicados na tabela de rótulos.")
rotulos = rotulos.set_index("id")
if not set(todos_ids).issubset(rotulos.index):
    raise ValueError("Há IDs selecionados sem rótulos.")
print(f"50% — {quantidade:,} imagens disponíveis; 4.000 IDs localizados.", flush=True)
print("Pasta usada:", pasta_imagens, flush=True)

# Carrega os dois módulos desta cópia exata do repositório.
sys.path.insert(0, str(repo_textura / "src"))
sys.modules.pop("image_preprocessing", None)
extrator = runpy.run_path(str(repo_textura / "src/02_preprocess.py"))
colunas_textura = [f"lbp_{i}" for i in range(10)] + [
    f"glcm_{propriedade}_{estatistica}"
    for propriedade in extrator["GLCM_PROPERTIES"]
    for estatistica in ("mean", "std")
]
if len(colunas_textura) != 20:
    raise ValueError("O extrator precisa gerar 20 características de textura.")
pacote = trabalho / "pacote"
pacote.mkdir()
(pacote / "splits").mkdir()
(pacote / "src").mkdir()


def validar_matriz(caminho, ids):
    matriz = pd.read_csv(caminho, dtype={"id": str}, keep_default_na=False)
    if matriz.columns.tolist() != ["id"] + colunas_textura:
        raise ValueError(f"Colunas incorretas: {caminho.name}")
    if matriz["id"].tolist() != ids:
        raise ValueError(f"Ordem ou quantidade de IDs incorreta: {caminho.name}")
    if not np.isfinite(matriz[colunas_textura].to_numpy(dtype=float)).all():
        raise ValueError(f"Valor ausente ou infinito: {caminho.name}")
    lbp = matriz[[f"lbp_{i}" for i in range(10)]].to_numpy(dtype=float)
    if (lbp < 0).any() or not np.allclose(lbp.sum(axis=1), 1.0, atol=1e-6, rtol=0):
        raise ValueError(f"Histograma LBP inválido: {caminho.name}")
    return matriz


with tqdm(total=4000, desc="Extração LBP/GLCM", unit="imagem") as barra:
    for nome, ids in splits.items():
        linhas = []
        for id_ in ids:
            imagem = extrator["preprocess_image"](pasta_imagens / f"{id_}.png")
            features = extrator["extract_features"](imagem)
            linhas.append({"id": id_, **{c: features[c] for c in colunas_textura}})
            barra.update(1)
        destino = pacote / f"texture_{nome}.csv"
        pd.DataFrame(linhas, columns=["id"] + colunas_textura).to_csv(destino, index=False)
        validar_matriz(destino, ids)
        meta = rotulos.loc[ids, ["boneage", "male"]].reset_index()
        if meta["id"].tolist() != ids:
            raise ValueError(f"Rótulos desalinhados: {nome}")
        meta.to_csv(pacote / f"metadata_{nome}.csv", index=False)
        shutil.copy2(repo_textura / "data/splits" / f"{nome}_ids.csv", pacote / "splits")

for nome, ids in splits.items():
    validar_matriz(pacote / f"texture_{nome}.csv", ids)
    print(f"{nome}: {len(ids)} × 20 características; ordem dos IDs conferida.", flush=True)
print("75% — Matrizes geradas e verificadas.", flush=True)

shutil.copy2(repo_textura / "data/splits/sample_ids.csv", pacote / "splits")
for nome in ("02_preprocess.py", "image_preprocessing.py"):
    shutil.copy2(repo_textura / "src" / nome, pacote / "src")

versoes = {
    nome: version(nome)
    for nome in ("numpy", "pandas", "Pillow", "scikit-image", "scipy",
                 "scikit-learn", "kagglehub", "tqdm")
}
# Registra as versões efetivamente carregadas, caso o runtime já tivesse imports.
versoes["numpy"] = np.__version__
versoes["pandas"] = pd.__version__
import PIL
import skimage
versoes["Pillow"] = PIL.__version__
versoes["scikit-image"] = skimage.__version__
(pacote / "requirements_extracao.txt").write_text(
    "\n".join(f"{nome}=={valor}" for nome, valor in versoes.items()) + "\n",
    encoding="utf-8",
)

instrucoes = """# Matrizes de textura RSNA

Treino: 2.800 × 20; validação: 600 × 20; teste: 600 × 20.
Cada CSV contém id + 20 características: 10 LBP e 10 GLCM.
As linhas seguem exatamente a ordem dos respectivos arquivos em splits/.
metadata_*.csv contém id, boneage (meses) e male na mesma ordem.
Semente da amostra e das divisões: 42. A extração é determinística.
Não recrie os splits nem reordene as matrizes sem reordenar os rótulos.

Pré-processamento comum: Pillow L; 224 × 224; Lanczos; float32 / 255.
Sem recorte, remoção de fundo, realce de contraste ou ajuste aprendido.
LBP: P=8, R=1, uniform; histograma normalizado com 10 bins.
GLCM: pixels reconstruídos por round(image * 255), uint8; quantização //16;
16 níveis; distâncias [1,2]; ângulos [0,π/4,π/2,3π/4]; symmetric=True;
normed=True; contrast, dissimilarity, homogeneity, energy, correlation;
média e desvio populacional entre as 8 combinações de distância/ângulo.
pixel_mean é calculado pelo script de origem, mas omitido deste pacote
para manter a matriz LBP/GLCM com 20 atributos. id não é atributo preditivo.
Sexo não está nas 20 colunas; use metadata_* para acrescentá-lo de forma
consistente às famílias conforme o protocolo definido pela equipe.

Uso após descompactar:
```python
import pandas as pd
matriz = pd.read_csv('texture_train.csv', dtype={'id': str})
meta = pd.read_csv('metadata_train.csv', dtype={'id': str})
assert matriz['id'].tolist() == meta['id'].tolist()
X_train = matriz.drop(columns='id').to_numpy(dtype=float)
y_train = meta['boneage'].to_numpy(dtype=float)
```
Scaler/PCA/seleção de atributos devem ser ajustados apenas no treino ou
no treino de cada dobra. Reserve o teste para avaliação final única.

Limitação: IDs de imagem distintos não comprovam separação por paciente;
a base usada não fornece uma chave verificável de paciente. A equipe deve
registrar e revisar essa limitação metodológica.

O notebook usa as funções do script em src/ e seleciona somente lbp_*/glcm_*.
O script de origem via CLI gera também pixel_mean; para reproduzir o pacote,
remova essa coluna dos CSVs de 21 características gerados por ele.
manifest.json registra commit, parâmetros, versões e SHA-256 dos arquivos.
Este pacote entrega características e metadados, sem as radiografias.
"""
(pacote / "README.md").write_text(instrucoes, encoding="utf-8")
manifesto = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repository": REPO_URL, "commit": COMMIT, "dataset": DATASET,
    "sample_seed": 42, "split_seed": 42,
    "rows": TAMANHOS, "feature_names": colunas_textura,
    "excluded_features": ["pixel_mean"], "id_order": "exactly as frozen split CSV",
    "python": platform.python_version(), "packages": versoes,
    "preprocessing": {"gray": "Pillow L", "size": [224, 224],
                      "resize": "Lanczos", "normalize": "float32 / 255",
                      "crop": False, "background_removal": False},
    "lbp": {"P": 8, "R": 1, "method": "uniform", "bins": 10},
    "glcm": {"levels": 16, "distances": [1, 2], "angles_degrees": [0, 45, 90, 135],
             "symmetric": True, "normed": True, "std_ddof": 0,
             "properties": list(extrator["GLCM_PROPERTIES"])},
    "patient_separation": "not_verifiable",
    "files_sha256": {
        str(p.relative_to(pacote)): sha256(p)
        for p in sorted(pacote.rglob("*")) if p.is_file()
    },
}
(pacote / "manifest.json").write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False), encoding="utf-8"
)
zip_saida = trabalho / "matrizes_textura_RSNA_4000.zip"
with ZipFile(zip_saida, "w", compression=ZIP_DEFLATED) as arquivo_zip:
    for caminho in sorted(pacote.rglob("*")):
        if caminho.is_file():
            arquivo_zip.write(caminho, str(caminho.relative_to(pacote)))
with ZipFile(zip_saida) as arquivo_zip:
    defeito = arquivo_zip.testzip()
    if defeito is not None:
        raise RuntimeError(f"ZIP com erro de integridade: {defeito}")
    for nome, esperado in manifesto["files_sha256"].items():
        if hashlib.sha256(arquivo_zip.read(nome)).hexdigest() != esperado:
            raise RuntimeError(f"Conteúdo divergente no ZIP: {nome}")

print("100% do processamento — ZIP preparado e conferido.", flush=True)
print("Arquivo:", zip_saida, flush=True)
print("SHA-256 do ZIP:", sha256(zip_saida), flush=True)
print("Pacote gerado. O compartilhamento do ZIP é uma etapa externa ao processamento.", flush=True)
from google.colab import files
files.download(str(zip_saida))


0% — Preparando uma execução independente das células antigas.
25% — Código e IDs congelados preparados.


100%|██████████| 9.29G/9.29G [01:39<00:00, 101MB/s]

Extracting files...


50% — 12,611 imagens disponíveis; 4.000 IDs localizados.
Pasta usada: /root/.cache/kagglehub/datasets/kmader/rsna-bone-age/versions/2/boneage-training-dataset/boneage-training-dataset


Extração LBP/GLCM:   0%|          | 0/4000 [00:00<?, ?imagem/s]

train: 2800 × 20 características; ordem dos IDs conferida.
val: 600 × 20 características; ordem dos IDs conferida.
test: 600 × 20 características; ordem dos IDs conferida.
75% — Matrizes geradas e verificadas.
100% do processamento — ZIP preparado e conferido.
Arquivo: /content/entrega_textura_cdt6vorb/matrizes_textura_RSNA_4000.zip
SHA-256 do ZIP: 8ae70870ae83c7e7c3a9ade89eb66e1d3695277c13b86b3c11c45a97c764d64b


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>